One subtle but important detail (ESCO-specific). In your dataset: 
- Many rows share the same occupationUri

Therefore, many rows share the same jd_text. This means:
- during evaluation, there may be multiple identical JDs in the corpus
- during training, identical jd_text may appear across different batches

This is not wrong, but you must be consistent:

Option A (simplest, acceptable for thesis)
- Treat each row independently
- One CV → one JD instance
- Report metrics as-is

Option B (cleaner retrieval framing, optional)
- Build corpus as unique JDs by occupationUri
- Map each CV query to its correct occupationUri
- Evaluate retrieval over unique jobs

If you want, I can help you implement Option B later (many theses don’t).

<a class="anchor" id="chapter1"></a>

# 1. Imports

</a>

In [1]:
import pandas as pd
import numpy as np
import os
import math

In [2]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from torch.utils.data import DataLoader

/opt/anaconda3/envs/ResumeMatcherThesis/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<a class="anchor" id="chapter2"></a>

# 2. Load Data

</a>

In [3]:
train = pd.read_csv("/Users/chloedeschanel/Desktop/NOVA IMS/Thesis/Dataset/Processed/train_pairs.csv")
val = pd.read_csv("/Users/chloedeschanel/Desktop/NOVA IMS/Thesis/Dataset/Processed/val_pairs.csv")
test = pd.read_csv("/Users/chloedeschanel/Desktop/NOVA IMS/Thesis/Dataset/Processed/test_pairs.csv")

<a class="anchor" id="chapter3"></a>

# 3. Initial Analysis

</a>

Each individual contributes multiple training instances corresponding to successive career transitions. As a result, person identifiers may appear multiple times, with each row representing a distinct CV-history–to–job transition.

In [4]:
train.head()

,person_id,t,cv_codes,cv_labels,cv_text,target_code,target_label,target_label_key,preferredLabel_key,jd_text,occupationUri,iscoGroup
0,0,1,['2353.1'],['language school teacher'],Work experience: language school teacher,3512.1,ICT help desk agent,ict help desk agent,ict help desk agent,ICT help desk agent. ICT help desk agents prov...,http://data.europa.eu/esco/occupation/aaeec9a7...,3512
1,0,2,"['2353.1', '3512.1']","['language school teacher', 'ICT help desk age...",Work experience: language school teacher; ICT ...,3512.2,ICT help desk manager,ict help desk manager,ict help desk manager,ICT help desk manager. ICT help desk managers ...,http://data.europa.eu/esco/occupation/1242d99a...,3512
2,0,3,"['2353.1', '3512.1', '3512.2']","['language school teacher', 'ICT help desk age...",Work experience: language school teacher; ICT ...,4222.1,customer contact centre information clerk,customer contact centre information clerk,customer contact centre information clerk,customer contact centre information clerk. Cus...,http://data.europa.eu/esco/occupation/b7b75eb6...,4222
3,0,4,"['2353.1', '3512.1', '3512.2', '4222.1']","['language school teacher', 'ICT help desk age...",Work experience: language school teacher; ICT ...,1324.4,move manager,move manager,move manager,move manager. Move managers coordinate all the...,http://data.europa.eu/esco/occupation/8f928d49...,1324
4,0,5,"['2353.1', '3512.1', '3512.2', '4222.1', '1324...","['language school teacher', 'ICT help desk age...",Work experience: language school teacher; ICT ...,3118.3.12,product development engineering drafter,product development engineering drafter,product development engineering drafter,product development engineering drafter. Produ...,http://data.europa.eu/esco/occupation/c52129d9...,3118


In [5]:
print("Unique targets:", train["target_label"].nunique())
print("Total pairs:", len(train))

freq = train["target_label"].value_counts()
print(freq.describe())

Unique targets: 2912
Total pairs: 707633
count     2912.000000
mean       243.005838
std       1115.434370
min          1.000000
25%         13.000000
50%         41.000000
75%        141.250000
max      31548.000000
Name: count, dtype: float64


In [6]:
train["target_label"].value_counts().head(10)

target_label
administrative assistant    31548
sales assistant             23027
shop assistant              18102
warehouse worker            16442
cashier                     12617
kitchen assistant           12023
waiter/waitress             10964
building cleaner            10717
factory hand                 9893
warehouse order picker       8791
Name: count, dtype: int64

<a class="anchor" id="chapter3"></a>

# 3. Negative Sampling

</a>

In [7]:
# Build training examples (positive pairs only)
def build_pairs_df(df, max_samples=None):

    """
    Converts a pandas DataFrame with columns [cv_text, jd_text] into a list of Sentence-Transformers InputExample objects.
    Each row is a *positive* pair: (anchor = cv_text, positive = jd_text)
    MultipleNegativesRankingLoss uses *in-batch negatives* to add negatives automatically:
    - in each training batch, all other jd_text are treated as negatives for a given cv_text.
    """

    if max_samples is not None:
        df = df.head(max_samples)

    examples = []
    for _, row in df.iterrows():
        cv = row["cv_text"]
        jd = row["jd_text"]

        # Basic filtering: ensure non-empty strings
        if isinstance(cv, str) and isinstance(jd, str) and cv.strip() and jd.strip():
            examples.append(InputExample(texts=[cv, jd]))
    return examples

In [8]:
# Build retrieval evaluator on validation split
def build_ir_eval_unique_jobs_df(df, name="val-ir-unique-jobs"):

    """
    Builds an InformationRetrievalEvaluator for CV -> JD retrieval.

    Deduplicate the corpus by occupationUri:
    - In the dataset, many rows can share the same occupationUri, which typically means the same jd_text is repeated.
    - If we evaluated with one JD per row, duplicates of the *correct* jd_text would appear as "negatives", artificially hurting MRR/nDCG.
    - Using unique occupationUri gives a cleaner "retrieve the right job" framing and a fairer metric.

    Evaluation setup:
    - Queries: one query per row (cv_text)
    - Corpus: unique JDs keyed by occupationUri
    - Relevant doc(s) for each query: the single occupationUri from that row
    """

    # Corpus: {occupationUri: jd_text} (unique jobs)
    corpus = (
        df[["occupationUri", "jd_text"]]
        .dropna()
        .drop_duplicates(subset=["occupationUri"])
        .set_index("occupationUri")["jd_text"]
        .to_dict()
    )

    queries = {} # {qid: cv_text}
    relevant_docs = {} # {qid: set([occupationUri])}

    # Queries: one per row, mapped to the relevant occupationUri
    q_idx = 0
    for _, row in df.iterrows():
        cv = row.get("cv_text")
        occ = row.get("occupationUri")

        # Skip invalid rows
        if not isinstance(cv, str) or not cv.strip():
            continue
        if not isinstance(occ, str) or occ not in corpus:
            continue

        qid = f"q{q_idx}"
        queries[qid] = cv
        relevant_docs[qid] = {occ}
        q_idx += 1

    return InformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        name=name,
        precision_recall_at_k=[1, 3, 5, 10, 20], # Recall@K: is the correct job in the top K?
        mrr_at_k=[10], # MRR@K: how early does the correct job appear (rank-sensitive)?
        ndcg_at_k=[10], # nDCG@K: rank-sensitive gain (also common in IR)
        map_at_k=[10], # MAP@K: average precision within top K
    )

In [9]:
# Prepare train/val/test from pandas dataframes
train_df = train
val_df = val
test_df = test

In [10]:
# Build training examples from TRAIN ONLY (avoid leakage)
train_examples = build_pairs_df(train_df)

In [11]:
# Build evaluator on validation set (used for monitoring training)
ir_evaluator = build_ir_eval_unique_jobs_df(val_df, name="val-ir")

In [12]:
print("Train examples:", len(train_examples))
print("Val queries:", len(ir_evaluator.queries))
print("Val corpus (unique jobs):", len(ir_evaluator.corpus))

Train examples: 707633
Val queries: 88489
Val corpus (unique jobs): 2501


In [13]:
# Train SBERT baseline with in-batch negatives
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=128, drop_last=True)
    # drop_last=True makes every batch the same size to help with in-batch negatives behave consistently

In [14]:
# MultipleNegativesRankingLoss: For each (cv, jd_pos) in the batch, all other jd_pos from the batch are treated as negatives.
train_loss = losses.MultipleNegativesRankingLoss(model)

In [ ]:
epochs = 1

# Helps stabilize early training (especially with transformers + AdamW)
warmup_steps = math.ceil(len(train_dataloader) * epochs * 0.1)

In [16]:
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=ir_evaluator,
    epochs=epochs,
    warmup_steps=warmup_steps,
    # Evaluate periodically during training:
    evaluation_steps=max(1000, len(train_dataloader)//2),
    output_path="sbert_stage1_baseline",
    show_progress_bar=True
)

Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [17]:
import torch
torch.cuda.is_available()

False

In [18]:
torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

'CPU'